# Run Summary

Notebook genérico para sumarizar todas as runs (`cfg*/metrics.json`) de um experimento.

**Parâmetro:** `RUN_DIR` — caminho para o diretório de uma execução (`results/YYYY-MM-DD_HHMMSS`).

Pode ser sobrescrito via variável de ambiente `RUN_DIR`.

Saídas geradas dentro de `RUN_DIR`:
- `run_summary.csv` — todas as runs (linha por cfg) com métricas e principais hiperparâmetros
- `run_summary_by_model_mode.csv` — estatísticas agregadas por (model, mode)
- `run_summary_top10.csv` — top 10 configs por `test_f1_macro`
- `run_summary_best_per_model.csv` — melhor cfg por modelo
- `run_summary_boxplot.png`, `run_summary_best_bar.png` — visualizações

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

In [ ]:
# Parâmetro: RUN_DIR (sobrescrito via env var)
RUN_DIR = os.environ.get('RUN_DIR', 'ford-a/results/2026-05-10_152016')
RUN_DIR = Path(RUN_DIR).resolve()
assert RUN_DIR.exists(), f'RUN_DIR não encontrado: {RUN_DIR}'
print(f'RUN_DIR: {RUN_DIR}')

## 1. Coleta de métricas
Varre todos os `*/cfg*/metrics.json` dentro de `RUN_DIR`.

In [ ]:
metric_files = sorted(RUN_DIR.glob('*/cfg*/metrics.json'))
print(f'metrics.json encontrados: {len(metric_files)}')
assert len(metric_files) > 0, 'Nenhum metrics.json encontrado.'

In [ ]:
# Hiperparâmetros escalares que queremos preservar (best-effort).
CONFIG_SCALAR_KEYS = [
    'learning_rate', 'batch_size', 'dropout_rate', 'l2_reg',
    'epochs', 'early_stopping_patience', 'levels', 'kernel_size',
    'patch_size', 'embed_dim', 'num_heads', 'ff_dim', 'num_layers',
    'reduce_lr_patience', 'reduce_lr_factor',
]

def _to_scalar(v):
    if isinstance(v, (list, tuple)):
        try:
            return ','.join(str(x) for x in v)
        except Exception:
            return str(v)
    if isinstance(v, dict):
        return json.dumps(v, sort_keys=True)
    return v

rows = []
for mf in metric_files:
    try:
        d = json.loads(mf.read_text())
    except Exception as e:
        print(f'erro lendo {mf}: {e}')
        continue
    cfg = d.get('config', {}) or {}
    row = {k: v for k, v in d.items() if k != 'config'}
    # Caminho relativo útil para joins
    row['_metric_path'] = str(mf.relative_to(RUN_DIR))
    row['model_mode'] = f"{d.get('model_name', '?')}_{d.get('mode', '?')}"
    # hiperparâmetros
    for k in CONFIG_SCALAR_KEYS:
        if k in cfg:
            row[f'cfg_{k}'] = _to_scalar(cfg[k])
    rows.append(row)

df = pd.DataFrame(rows)
print(f'Linhas: {len(df)}')
print(f'Colunas: {len(df.columns)}')
df.head()

In [ ]:
# Ordenar colunas: identificação → métricas test → val → train → cfg
id_cols = [c for c in ['model_name', 'mode', 'model_mode', 'config_idx', 'epochs_trained', '_metric_path'] if c in df.columns]
test_cols = [c for c in df.columns if c.startswith('test_')]
val_cols = [c for c in df.columns if c.startswith('val_')]
train_cols = [c for c in df.columns if c.startswith('train_')]
cfg_cols = sorted([c for c in df.columns if c.startswith('cfg_')])
other_cols = [c for c in df.columns if c not in id_cols + test_cols + val_cols + train_cols + cfg_cols]

df = df[id_cols + test_cols + val_cols + train_cols + cfg_cols + other_cols]
df.head()

## 2. CSV completo
Persiste o sumário com todas as runs e métricas em `run_summary.csv`.

In [ ]:
summary_csv = RUN_DIR / 'run_summary.csv'
df.to_csv(summary_csv, index=False)
print(f'CSV salvo: {summary_csv}  ({summary_csv.stat().st_size/1024:.1f} KB)')

## 3. Insights por (model, mode)
Estatísticas agregadas: média, std, máximo, contagem por combinação.

In [ ]:
# Detecção automática de tipo de tarefa (classificação vs regressão)
if 'test_f1_macro' in df.columns:
    TASK_TYPE = 'classification'
    PRIMARY_METRIC = 'test_f1_macro'
    HIGHER_BETTER = True
elif 'val_f1_macro' in df.columns:
    TASK_TYPE = 'classification'
    PRIMARY_METRIC = 'val_f1_macro'
    HIGHER_BETTER = True
elif 'test_r2' in df.columns:
    TASK_TYPE = 'regression'
    PRIMARY_METRIC = 'test_r2'
    HIGHER_BETTER = True
elif 'test_rmse' in df.columns:
    TASK_TYPE = 'regression'
    PRIMARY_METRIC = 'test_rmse'
    HIGHER_BETTER = False
else:
    TASK_TYPE = 'unknown'
    PRIMARY_METRIC = None
    HIGHER_BETTER = True

# AGG_METRICS = todas as métricas test_/val_ presentes
candidate = ['test_accuracy', 'test_f1_macro', 'test_f1_weighted', 'test_auc_roc',
             'test_mse', 'test_rmse', 'test_mae', 'test_r2',
             'val_accuracy', 'val_f1_macro', 'val_auc_roc',
             'val_mse', 'val_rmse', 'val_mae', 'val_r2']
AGG_METRICS = [m for m in candidate if m in df.columns]
print(f'Tarefa: {TASK_TYPE}')
print(f'Métrica principal: {PRIMARY_METRIC}  (maior = melhor: {HIGHER_BETTER})')
print(f'Métricas agregadas: {AGG_METRICS}')

In [ ]:
by_mm = df.groupby(['model_name', 'mode'])[AGG_METRICS].agg(['count', 'mean', 'std', 'max']).round(4)
by_mm.columns = ['_'.join(c) for c in by_mm.columns]
by_mm = by_mm.reset_index()
by_mm_csv = RUN_DIR / 'run_summary_by_model_mode.csv'
by_mm.to_csv(by_mm_csv, index=False)
print(f'Agregado por (model, mode) salvo: {by_mm_csv}')
by_mm.sort_values(f'{PRIMARY_METRIC}_max' if PRIMARY_METRIC else AGG_METRICS[0] + '_max', ascending=False).head(15)

## 4. Top 10 configs (por métrica principal)

In [ ]:
top10 = df.sort_values(PRIMARY_METRIC, ascending=not HIGHER_BETTER).head(10).copy() if PRIMARY_METRIC else df.head(10).copy()
top10_csv = RUN_DIR / 'run_summary_top10.csv'
top10.to_csv(top10_csv, index=False)
print(f'Top 10 salvo: {top10_csv}')
top10[['model_name', 'mode', 'config_idx'] + AGG_METRICS]

## 5. Melhor configuração por modelo

In [ ]:
if PRIMARY_METRIC:
    if HIGHER_BETTER:
        idx_best = df.groupby(['model_name', 'mode'])[PRIMARY_METRIC].idxmax()
    else:
        idx_best = df.groupby(['model_name', 'mode'])[PRIMARY_METRIC].idxmin()
    best = df.loc[idx_best].sort_values(PRIMARY_METRIC, ascending=not HIGHER_BETTER).reset_index(drop=True)
else:
    best = df.drop_duplicates(['model_name', 'mode']).reset_index(drop=True)
best_csv = RUN_DIR / 'run_summary_best_per_model.csv'
best.to_csv(best_csv, index=False)
print(f'Melhores por (model, mode) salvo: {best_csv}')
best[['model_name', 'mode', 'config_idx'] + AGG_METRICS]

## 6. Visualizações

In [ ]:
if PRIMARY_METRIC is not None:
    fig, ax = plt.subplots(figsize=(14, 6))
    order = df.groupby('mode')[PRIMARY_METRIC].median().sort_values(ascending=not HIGHER_BETTER).index.tolist()
    sns.boxplot(data=df, x='mode', y=PRIMARY_METRIC, order=order, ax=ax)
    sns.stripplot(data=df, x='mode', y=PRIMARY_METRIC, order=order, color='black', alpha=0.25, size=2, ax=ax)
    ax.set_title(f'Distribuição de {PRIMARY_METRIC} por modo  —  {RUN_DIR.name}')
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    fig.savefig(RUN_DIR / 'run_summary_boxplot.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
if PRIMARY_METRIC is not None:
    fig, ax = plt.subplots(figsize=(14, 6))
    plot_best = best.sort_values(PRIMARY_METRIC, ascending=HIGHER_BETTER)
    labels = plot_best['model_name'] + ' / ' + plot_best['mode']
    colors = sns.color_palette('viridis', len(plot_best))
    ax.barh(labels, plot_best[PRIMARY_METRIC], color=colors)
    ax.set_xlabel(PRIMARY_METRIC)
    ax.set_title(f'Melhor {PRIMARY_METRIC} por (model, mode)  —  {RUN_DIR.name}')
    for i, v in enumerate(plot_best[PRIMARY_METRIC]):
        ax.text(v, i, f' {v:.4f}', va='center', fontsize=9)
    plt.tight_layout()
    fig.savefig(RUN_DIR / 'run_summary_best_bar.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Insights textuais

In [ ]:
print(f'== Resumo — {RUN_DIR.name} ==')
print(f'Total de runs (cfg): {len(df)}')
print(f'Modelos: {sorted(df["model_name"].unique())}')
print(f'Modos: {sorted(df["mode"].unique())}')
print(f'Combinações (model, mode): {df.groupby(["model_name", "mode"]).ngroups}')
if PRIMARY_METRIC is not None:
    top_row = df.loc[df[PRIMARY_METRIC].idxmax() if HIGHER_BETTER else df[PRIMARY_METRIC].idxmin()]
    print(f'\nMelhor cfg global por {PRIMARY_METRIC}: {top_row[PRIMARY_METRIC]:.4f}')
    print(f'  modelo = {top_row["model_name"]} | modo = {top_row["mode"]} | cfg = {top_row["config_idx"]}')

    print('\nMédia de test_f1_macro por modo (ranqueado):')
    mean_by_mode = df.groupby('mode')[PRIMARY_METRIC].agg(['mean', 'std', 'max', 'count']).round(4)
    print(mean_by_mode.sort_values('mean', ascending=not HIGHER_BETTER))

    print('\nMelhor cfg por modelo:')
    print(best[['model_name', 'mode', 'config_idx', PRIMARY_METRIC]].to_string(index=False))

    # Comparação learned vs db4 vs raw (média geral)
    print('\nMédia geral por categoria de modo:')
    def _cat(m):
        if 'learned' in m: return 'learned'
        if 'db4' in m: return 'db4'
        if 'raw' in m: return 'raw'
        return 'other'
    df['_mode_cat'] = df['mode'].apply(_cat)
    print(df.groupby('_mode_cat')[PRIMARY_METRIC].agg(['mean', 'std', 'max', 'count']).round(4))